In [4]:
%%capture
import os
import sys


!{sys.executable} -m pip install pip3-autoremove
!{sys.executable} -m pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!{sys.executable} -m pip install unsloth
!{sys.executable} -m pip install transformers==4.56.2
!{sys.executable} -m pip install --no-deps trl==0.22.2

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA H100 80GB HBM3


In [2]:
from huggingface_hub import login

login()

In [11]:
%uv pip install wandb


Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 12ms
Note: you may need to restart the kernel to use updated packages.


In [3]:
from unsloth import PatchDPOTrainer

PatchDPOTrainer()

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [8]:
from unsloth import FastLanguageModel
import torch


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "NgVietAnh41/Qwen3-4b-it-final-VietMedQA",
    max_seq_length=2048,
    dtype = None,
    load_in_4bit=True,
    token=hf_token
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.179 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 9.0. CUDA Toolkit: 12.9. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
import json
import datasets
from datasets import Dataset
import random

data_list = []
with open("DPOdataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        data_list.append(data)

dataset = Dataset.from_list(data_list) 


In [10]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [11]:
dataset[0]

{'prompt': 'Xin chào bác sĩ,\nHiện tôi hay bị buồn bực 2 bên đùi ,tối nào trước khi đi ngủ cũng chỉ muốn mát xa,đấm mạnh vào đùi khu vực xương đùi,đùi non và giữa đùi ,hiện tượng này kéo dài rất lâu rồi.ko biết là do đâu',
 'chosen': 'Chào ban.\nHiện tại cân nặng của bạn như thế nào ? Chế độ ăn uống sinh hoạt ra sao ? Qua mô tả , những biểu hiện này có thể do cơ thể bạn thiếu một số vi chất. Bạn nên bổ sung đầy đủ chất dinh dưỡng trong khẩu phần ăn, đặc biệt là Canxi nhé. \nThân mến.',
 'rejected': 'Chào bạn,\nTình trạng của bạn có thể do căng cơ gây nên. Trước khi đi ngủ bạn có thể vận động nhẹ nhàng, xoa bóp và ngày sau đi bơi hoặc đi bộ giúp cải thiện.\nNgoài ra, bạn nên bổ sung thêm nhóm vi chất B bằng cách uống sữa chua, ăn rau xanh và hoa quả tươi.\nChế độ sinh hoạt và nghỉ ngơi cũng cần phải khoa học, điều độ, tránh áp lực trong công việc cũng như các mối quan hệ xã giao, nên tìm được một người bạn cùng hoạt động thể thao để tạo sức khỏe với người thân và bạn bè.\nThân mến!'}

In [12]:
def format_dpo_func(example):
    prompt_messages = [
        {"role": "user", "content": example['prompt']} 
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        prompt_messages, 
        tokenize = False, 
        add_generation_prompt = True 
    )
    return {
        "prompt": formatted_prompt,      
        "chosen": example["chosen"],     
        "rejected": example["rejected"], 
    }

In [13]:
data = dataset.train_test_split(test_size=0.1, shuffle=True, seed = 42)



In [14]:
train_data = data['train']
test_data = data['test']

In [15]:
train_data[0]

{'prompt': 'Xin chào bác sĩ, tôi muốn được tư vấn về loại thuốc tăng kích thước dương vật và có tác dụng tốt cho sinh lí. Kính mong được tư vấn! Tôi xin cảm ơn!',
 'chosen': 'Chào bạn\nHiện nay trên thị trường có rất nhiều loại thuốc được quảng cáo là làm tăng kích thước dương vật nhưng hầu hết đều không được kiểm chứng về chất lượng. Vì thế bác sỹ khuyên bạn không nên dùng thuốc mà tìm đến những phương pháp làm tăng kích thước dương vật từ tự nhiên. Tuy nhiên cần kiên trì thực hiện. \n- Các nguồn thực phẩm tốt có thể kể đến như: hàu, chuối, thịt dê, bò, cua biển, chạch, nước ép táo, dầu oliu…rất tốt cho sức khỏe nam giới cũng như giúp dương vật khỏe mạnh thúc đẩy quá trình phát triển kích thước.\n- Tắm nước nóng trước khi sinh hoạt tình dục, mỗi khi lâm trận hãy tắm trước bằng nước nóng việc này sẽ giúp các mạch máu của dương vật nở ra, giúp lưu thông máu nhanh chóng dễ dàng từ đó khi hoạt sinh hoạt tình dục diễn ra cũng là lúc tập luyện cho dương vật.\n- Dùng 2 ngón cái và ngón trỏ b

In [16]:
train_data = train_data.map(format_dpo_func)
test_data = test_data.map(format_dpo_func)

Map:   0%|          | 0/2700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [17]:
train_data[0]

{'prompt': '<|im_start|>user\nXin chào bác sĩ, tôi muốn được tư vấn về loại thuốc tăng kích thước dương vật và có tác dụng tốt cho sinh lí. Kính mong được tư vấn! Tôi xin cảm ơn!<|im_end|>\n<|im_start|>assistant\n',
 'chosen': 'Chào bạn\nHiện nay trên thị trường có rất nhiều loại thuốc được quảng cáo là làm tăng kích thước dương vật nhưng hầu hết đều không được kiểm chứng về chất lượng. Vì thế bác sỹ khuyên bạn không nên dùng thuốc mà tìm đến những phương pháp làm tăng kích thước dương vật từ tự nhiên. Tuy nhiên cần kiên trì thực hiện. \n- Các nguồn thực phẩm tốt có thể kể đến như: hàu, chuối, thịt dê, bò, cua biển, chạch, nước ép táo, dầu oliu…rất tốt cho sức khỏe nam giới cũng như giúp dương vật khỏe mạnh thúc đẩy quá trình phát triển kích thước.\n- Tắm nước nóng trước khi sinh hoạt tình dục, mỗi khi lâm trận hãy tắm trước bằng nước nóng việc này sẽ giúp các mạch máu của dương vật nở ra, giúp lưu thông máu nhanh chóng dễ dàng từ đó khi hoạt sinh hoạt tình dục diễn ra cũng là lúc tập 

In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Currently only supports dropout = 0
    bias = "none",    # Currently only supports bias = "none"
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [19]:
from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 2,
        learning_rate = 2e-6,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.05,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
        eval_strategy = "steps", # Bắt buộc phải có dòng này mới test được
        eval_steps = 10,         # Cứ 10 bước train thì dừng lại chấm điểm tập test 1 lần
    ),
    beta = 0.1,
    train_dataset = train_data,
    eval_dataset = test_data,
    tokenizer = tokenizer,
    max_length = 2048,
    max_prompt_length = 1024,
)

Extracting prompt in train dataset (num_proc=64):   0%|          | 0/2700 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=64):   0%|          | 0/2700 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=64):   0%|          | 0/2700 [00:00<?, ? examples/s]

Extracting prompt in eval dataset (num_proc=64):   0%|          | 0/300 [00:00<?, ? examples/s]

Applying chat template to eval dataset (num_proc=64):   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing eval dataset (num_proc=64):   0%|          | 0/300 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [20]:
dpo_trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,700 | Num Epochs = 2 | Total steps = 676
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
10,0.723400,0.687583,-0.000241,-0.013795,0.523333,0.013555,-151.526413,-206.146500,-4.110187,-4.165970,0,0,0
20,0.683000,0.691235,-0.002292,-0.008917,0.526667,0.006625,-151.546936,-206.097717,-4.110139,-4.166224,No Log,No Log,No Log
30,0.691100,0.689225,-0.002672,-0.013222,0.550000,0.010550,-151.550735,-206.140762,-4.109308,-4.165790,No Log,No Log,No Log
40,0.699300,0.688449,-0.004984,-0.017150,0.530000,0.012166,-151.573868,-206.180038,-4.109463,-4.165239,No Log,No Log,No Log
50,0.700100,0.692075,-0.004405,-0.008733,0.543333,0.004328,-151.568069,-206.095856,-4.109529,-4.165427,No Log,No Log,No Log
60,0.673200,0.690725,-0.007047,-0.014543,0.496667,0.007497,-151.594482,-206.153961,-4.109418,-4.165709,No Log,No Log,No Log
70,0.686000,0.689444,-0.005245,-0.015227,0.513333,0.009983,-151.576477,-206.160812,-4.109026,-4.165687,No Log,No Log,No Log
80,0.710200,0.679281,-0.002318,-0.032810,0.613333,0.030492,-151.547180,-206.336639,-4.108782,-4.165295,No Log,No Log,No Log
90,0.692400,0.682366,-0.000273,-0.025297,0.580000,0.025024,-151.526764,-206.261505,-4.106997,-4.163945,No Log,No Log,No Log
100,0.711300,0.680169,-0.002024,-0.030535,0.610000,0.028512,-151.544250,-206.313873,-4.106165,-4.163187,No Log,No Log,No Log


TrainOutput(global_step=676, training_loss=0.580027857123042, metrics={'train_runtime': 2608.9601, 'train_samples_per_second': 2.07, 'train_steps_per_second': 0.259, 'total_flos': 0.0, 'train_loss': 0.580027857123042, 'epoch': 2.0})

In [26]:
model.push_to_hub_merged("NgVietAnh41/Qwen3-4b-it-VietMedQA-final-DPO", tokenizer, save_method = "merged_16bit", token = hf_token)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


...

Unsloth: Merging weights into 16bit:  50%|███████████████████▌                   | 1/2 [00:50<00:50, 50.13s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|███████████████████████████████████████| 2/2 [01:17<00:00, 38.78s/it]


Unsloth: Merge process complete. Saved to `/root/NgVietAnh41/Qwen3-4b-it-VietMedQA-final-DPO`


In [28]:
%%capture
!apt-get update -y
!apt-get install -y libcurl4-openssl-dev

In [29]:
model.push_to_hub_gguf("NgVietAnh41/Qwen3-4b-it-VietMedQA-final-DPO-gguf", tokenizer, quantization_method = "q4_k_m", token = hf_token)

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `/tmp/unsloth_gguf_ni124sbf`: 100%|███████| 2/2 [00:03<00:00,  1.88s/it]


Successfully copied all 2 files from cache to `/tmp/unsloth_gguf_ni124sbf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_ni124sbf_gguf/Qwen3-4b-it-final-VietMedQA.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_ni124sbf_gguf/Qwen3-4b-it-final-VietMedQA.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'NgVietAnh41/Qwen3-4b-it-final-VietMedQA'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /tmp/unsloth_gguf_ni124sbf_gguf/Qwen3-4b-it-final-VietMedQA.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Uploading GGUF to Huggingface Hub...
Uploading Qwen3-4b-it-final-VietMedQA.Q4_K_M.gguf...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/NgVietAnh41/Qwen3-4b-it-VietMedQA-final-DPO-gguf
Unsloth: Cleaning up temporary files...


'NgVietAnh41/Qwen3-4b-it-VietMedQA-final-DPO-gguf'